### Get icd code YAML file

In [ ]:
#import libraries 

import pandas as pd
import openpyxl
import duckdb 
import pyarrow.parquet as pq

In [ ]:
# load and read icd codes datasets 

icd9_file = pd.read_excel('../data/input/icd_codes/icd9_codes.xlsx')
icd10_file = pd.read_excel('../data/input/icd_codes/icd10_codes.xlsx')

In [ ]:
# Function to match ICD codes

def match_codes(df, codes, exclude = []):
    pattern = f"^(?:{'|'.join(codes)})"
    matched = df[df['CODE'].astype(str).str.contains(pattern, regex=True)]

 # Exclude specific codes like 066.4
    if exclude:
        for prefix in exclude:
            matched = matched[~matched['CODE'].astype(str).str.startswith(prefix)]

    return matched['CODE'].unique()

In [ ]:
# outcomes and codes given by the researcher 

conditions = {
    "Dengue": {
        'icd9': ["061"],
        'icd10': ["A90", "A91"]
    },
    "West Nile Virus": {
        'icd9': ["0664"],
        'icd10': ["A923"]
    },
    "Arthropod-borne and Hemorrhagic Fevers": {
        'icd9': ["060", "064", "065", "066"],  # Excluding 066.4
        'icd10': ["A92", "A93", "A94", "A95", "A98", "A99"]  # Excluding A92.3
    },
    "Mosquito-borne Viral Encephalitis": {
        'icd9': ["062"],
        'icd10': ["A83"]
    },
    "Tick-borne Viral Encephalitis": {
        'icd9': ["063"],
        'icd10': ["A84"]
    },
    "Rickettsioses": {
        'icd9': ["080", "081", "082", "083"],
        'icd10': ["A75", "A76", "A77", "A78", "A79"]
    },
    "Lyme Disease": {
        'icd9': ["08881"],
        'icd10': ["A692"]
    },
    "Relapsing Fevers": {
        'icd9': ["087"],
        'icd10': ["A68"]
    },
    "Malaria": {
        'icd9': ["084"],
        'icd10': ["B50", "B51", "B52", "B53", "B54"]
    },
    "Leishmaniasis": {
        'icd9': ["085"],
        'icd10': ["B55"]
    },
    "Trypanosomiasis": {
        'icd9': ["086"],
        'icd10': ["B56", "B57"]
    },
    "Babesiosis": {
        'icd9': ["08882"],
        'icd10': ["B600"]
    },
    "Acute Respiratory Infections": {
        'icd9': [str(i) for i in range(460, 467)],  # 460-466
        'icd10': [f"J{str(i).zfill(2)}" for i in list(range(0, 7)) + list(range(20, 23))]  # J00-J06, J20-J22
    },
    "Influenza and Pneumonia": {
        'icd9': [str(i) for i in range(480, 489)],
        'icd10': [f"J{str(i).zfill(2)}" for i in range(9, 19)]  # J09-J18
    },
    "Tuberculosis": {
        'icd9': ["010", "011", "012"],
        'icd10': ["A15"]
    },
    "Intestinal Infectious Diseases": {
        'icd9': [str(i).zfill(3) for i in range(1, 10)],
        'icd10': [f"A{str(i).zfill(2)}" for i in range(0, 10)]
    },
    "Intestinal Viral Diseases": {
        'icd9': ["0086",str(9).zfill(3)],
        'icd10': [f"A{str(i).zfill(2)}" for i in range(8, 10)]  # A08–A09
    },
    "Intestinal Bacterial Diseases": {
        'icd9': [str(i).zfill(3) for i in range(1, 6)] + [f"008{d}" for d in range(0,6)],# 001–005, 0080–0085
        'icd10': [f"A{str(i).zfill(2)}" for i in range(0, 6)]   # A00–A05
    },
    "Clostridioides difficile": {
        "icd9": ["00845"],
        "icd10": ["A047"]
    }
}


In [ ]:
# define short names to be added to the yaml 

short_names = {
    "Dengue": "dengue",
    "West Nile Virus": "west_nile",
    "Arthropod-borne and Hemorrhagic Fevers": "arthropod_hemorrhagic_fever",
    "Mosquito-borne Viral Encephalitis": "mosquito_encephalitis",
    "Tick-borne Viral Encephalitis": "tick_encephalitis",
    "Rickettsioses": "rickettsioses",
    "Lyme Disease": "lyme",
    "Relapsing Fevers": "relapsing_fever",
    "Malaria": "malaria",
    "Leishmaniasis": "leishmaniasis",
    "Trypanosomiasis": "trypanosomiasis",
    "Babesiosis": "babesiosis",
    "Acute Respiratory Infections": "acute_resp_inf",
    "Influenza and Pneumonia": "flu_pneumonia",
    "Tuberculosis": "tb",
    "Intestinal Infectious Diseases": "intestinal_infect",
    "Intestinal Viral Diseases": "intestinal_viral_infect",
    "Intestinal Bacterial Diseases": "intestinal_bacterial_infect",
    "Clostridioides difficile" : "cdifficile"
}

In [ ]:
# write the outcomes names, icd codes 9 and 10 , outcome_criteria and readmission window days (if needed) to reflect in the yaml output of teh respective project 


yaml_output = " "

for condition, codes in conditions.items():

    identifier = short_names[condition]
    exclude_icd9 = [f"0664{i}" for i in range(10)] if condition == "Arthropod-borne and Hemorrhagic Fevers" else []
    exclude_icd10 = [f"A923{i}" for i in range(10)] if condition == "Arthropod-borne and Hemorrhagic Fevers" else []

    # Match ICD-9 and ICD-10 codes
    icd9_codes = match_codes(icd9_file, codes['icd9'], exclude=exclude_icd9) if codes['icd9'] else []
    icd10_codes = match_codes(icd10_file, codes['icd10'], exclude=exclude_icd10) if codes['icd10'] else []

    yaml_section = f"""
{identifier} :
    long_name : "{condition}"
    outcome_criteria: all
    readmission_window_days: 30
    icd9 : {list(icd9_codes)}
    icd9_count: {len(icd9_codes)}
    icd10 : {list(icd10_codes)}
    icd10_count: {len(icd10_codes)}
"""
    yaml_output += yaml_section

with open('../conf/icd_codes/michelle_tian_00.yml', 'w') as file:
    file.write(yaml_output)

print(yaml_output)